# Dissolve de Polígonos de Edifícios por Número de Polícia
**Problema:** A camada original de edifícios de Aveiro contém 88.649 polígonos, mas muitos deles
são fragmentos do mesmo edifício real (varandas, piscinas, anexos, zonas interiores).

**Solução:** Agrupar os polígonos com base nos pontos de número de polícia (),
que representam endereços postais reais, expandindo por adjacência geométrica para capturar
fragmentos contíguos sem endereço direto.

**Input:**   
**Output:**  (camada )

In [1]:
# =============================================================
# CÉLULA 1 — IMPORTAÇÕES
# =============================================================
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.strtree import STRtree
from shapely.ops import unary_union
from collections import defaultdict, Counter

print("Bibliotecas importadas com sucesso.")


Bibliotecas importadas com sucesso.


## 1. Carregar as Camadas do GeoPackage

In [2]:
GPKG = "MstCSCS_Sem_2526.gpkg"

# Camada de polígonos dos edifícios
gdf_ed = gpd.read_file(GPKG, layer="ed12_polygons_all_with_heights_clean")

# Camada de pontos de número de polícia (endereços reais)
gdf_np = gpd.read_file(GPKG, layer="avr_npolicia_pts")
gdf_np = gdf_np.to_crs(gdf_ed.crs)  # garantir mesmo CRS

# Camada de códigos postais já atribuídos
gdf_cp = gpd.read_file(GPKG, layer="postal_code_buildings_assigned")

print(f"Polígonos de edifícios : {len(gdf_ed):,}")
print(f"Pontos nº de polícia   : {len(gdf_np):,}")
print(f"Registos CP7           : {len(gdf_cp):,}")
print(f"CRS dos edifícios      : {gdf_ed.crs}")


Polígonos de edifícios : 88,649
Pontos nº de polícia   : 32,088
Registos CP7           : 17,559
CRS dos edifícios      : EPSG:3763


## 2. Passo 1 — Associar cada Ponto de Nº Polícia ao Polígono Correspondente
Para cada ponto de número de polícia:
1. Procura-se o polígono que **contém** o ponto (sobreposição direta)
2. Se não houver, aceita-se o polígono **mais próximo dentro de 10m** (tolerância para erros de posicionamento)
3. Se não houver nenhum dentro de 10m, o ponto fica sem polígono associado

O raio de 10m foi escolhido porque a análise exploratória mostrou que 91,2% dos pontos
ficam associados com este raio, com uma distância média de desvio de apenas 1,22m.

In [3]:
RAIO_MAX = 10  # metros

tree_ed = STRtree(gdf_ed.geometry.values)

# Para cada ponto de nº polícia → índice do polígono mais próximo
np_to_poly = {}  # npolicia_idx → poly_idx

for i, pt in enumerate(gdf_np.geometry):
    candidatos = tree_ed.query(pt.buffer(RAIO_MAX))
    # Prioridade 1: polígono que contém o ponto
    for c in candidatos:
        if gdf_ed.geometry.iloc[c].contains(pt):
            np_to_poly[i] = c
            break
    else:
        # Prioridade 2: polígono mais próximo dentro do raio
        melhor_dist, melhor_c = np.inf, None
        for c in candidatos:
            d = gdf_ed.geometry.iloc[c].distance(pt)
            if d <= RAIO_MAX and d < melhor_dist:
                melhor_dist, melhor_c = d, c
        if melhor_c is not None:
            np_to_poly[i] = melhor_c

# Inverter: poly_idx → lista de npolicia_idx associados
poly_to_nps = defaultdict(list)
for np_idx, poly_idx in np_to_poly.items():
    poly_to_nps[poly_idx].append(np_idx)

polys_com_np = set(poly_to_nps.keys())
polys_sem_np = set(range(len(gdf_ed))) - polys_com_np

print(f"Polígonos com nº polícia diretamente associado : {len(polys_com_np):,}")
print(f"Polígonos sem nº polícia (candidatos a fragmento): {len(polys_sem_np):,}")


Polígonos com nº polícia diretamente associado : 18,157
Polígonos sem nº polícia (candidatos a fragmento): 70,492


## 3. Passo 2 — Expandir por Adjacência Geométrica
Os polígonos sem número de polícia que sejam **adjacentes** (< 0,1m) a um polígono
com endereço são absorvidos pelo mesmo edifício. O processo itera até estabilizar
(sem mais polígonos a absorver).

Buffer de 0,1m para capturar fronteiras partilhadas com pequenos erros de digitalização.

In [5]:
tree_all = STRtree(gdf_ed.geometry.values)

# Inicializar: cada polígono com nº polícia é âncora do seu próprio grupo
poly_grupo = {p: p for p in polys_com_np}

# Expandir iterativamente até estabilizar
mudou = True
iteracao = 0
while mudou:
    mudou = False
    iteracao += 1
    for p in polys_sem_np:
        if p in poly_grupo:
            continue
        candidatos = tree_all.query(gdf_ed.geometry.iloc[p].buffer(0.1))
        for c in candidatos:
            if c != p and c in poly_grupo:
                poly_grupo[p] = poly_grupo[c]
                mudou = True
                break
    print(f"  Iteração {iteracao}: {len(poly_grupo):,} polígonos agrupados")

# Polígonos que não foram absorvidos = estruturas isoladas sem endereço
isolados = [p for p in range(len(gdf_ed)) if p not in poly_grupo]
print(f"Polígonos agrupados (com endereço ou adjacentes): {len(poly_grupo):,}")
print(f"Polígonos isolados (piscinas, muros, anexos soltos): {len(isolados):,}")


  Iteração 1: 55,672 polígonos agrupados
  Iteração 2: 67,201 polígonos agrupados
  Iteração 3: 69,915 polígonos agrupados
  Iteração 4: 70,682 polígonos agrupados
  Iteração 5: 70,920 polígonos agrupados
  Iteração 6: 70,997 polígonos agrupados
  Iteração 7: 71,030 polígonos agrupados
  Iteração 8: 71,036 polígonos agrupados
  Iteração 9: 71,041 polígonos agrupados
  Iteração 10: 71,041 polígonos agrupados
Polígonos agrupados (com endereço ou adjacentes): 71,041
Polígonos isolados (piscinas, muros, anexos soltos): 17,608


## 4. Passo 3 — Dissolve e Agregação de Atributos
Para cada grupo de polígonos:
- **Geometria:** união () de todos os fragmentos
- **Área:** soma das áreas individuais
- **Altura máxima:** máximo entre os fragmentos (representa o topo do edifício)
- **Altura média e cotas:** médias ponderadas pela área de cada fragmento
- **CP7:** código postal do fragmento com maior confiança de atribuição
- **Atributos representativos** (source_layer, src_file): do fragmento de maior área

In [6]:
# Preparar dicionário de CP7
gdf_cp_clean = (
    gdf_cp.sort_values("mean_assignment_confidence", ascending=False)
    .drop_duplicates("polygon_id")[["polygon_id", "cp7", "mean_assignment_confidence"]]
).copy()
gdf_cp_clean["polygon_id"] = gdf_cp_clean["polygon_id"].astype(int)
cp_dict = {
    int(r["polygon_id"]): {"cp7": r["cp7"], "conf": float(r["mean_assignment_confidence"])}
    for _, r in gdf_cp_clean.iterrows()
}

# Construir grupos finais
grupos_finais = defaultdict(list)
for p, g in poly_grupo.items():
    grupos_finais[g].append(p)
for p in isolados:
    grupos_finais[f"iso_{p}"].append(p)

# Agregar por grupo
registos = []
for grupo_id, idxs in grupos_finais.items():
    sub   = gdf_ed.iloc[idxs]
    geom  = unary_union(sub.geometry.values)
    pesos = sub["area_m2"] / sub["area_m2"].sum()
    rep   = sub.loc[sub["area_m2"].idxmax()]

    # Números de polícia associados ao grupo
    nps = list(set(np_idx for idx in idxs for np_idx in poly_to_nps.get(idx, [])))

    # CP7: fragmento com maior confiança
    orig_ids = sub["polygon_id"].tolist()
    matches  = [(pid, cp_dict[pid]["cp7"], cp_dict[pid]["conf"]) for pid in orig_ids if pid in cp_dict]
    cp7, cp7_conf = (None, None)
    if matches:
        best = max(matches, key=lambda x: x[2])
        cp7, cp7_conf = best[1], round(best[2], 3)

    registos.append({
        "building_id":      str(grupo_id),
        "n_fragmentos":     len(idxs),
        "n_npolicia":       len(nps),
        "tem_endereco":     len(nps) > 0,
        "polygon_ids_orig": ",".join(str(x) for x in orig_ids),
        "area_total_m2":    round(float(sub["area_m2"].sum()), 2),
        "altura_max_m":     round(float(sub["altura_edif_m"].max()), 2),
        "altura_mean_m":    round(float((sub["altura_edif_m"] * pesos).sum()), 2),
        "cota_topo_m":      round(float((sub["cota_topo_pt_m"] * pesos).sum()), 2),
        "cota_terreno_m":   round(float((sub["cota_terreno_est_m"] * pesos).sum()), 2),
        "source_layer":     rep["source_layer"],
        "src_file":         rep["src_file"],
        "has_height":       bool(sub["has_height"].any()),
        "cp7":              cp7,
        "cp7_confidence":   cp7_conf,
        "geometry":         geom,
    })

gdf_final = gpd.GeoDataFrame(registos, crs=gdf_ed.crs)
print(f"GeoDataFrame final criado: {len(gdf_final):,} edifícios")


GeoDataFrame final criado: 35,765 edifícios


## 5. Estatísticas Finais e Exportação

In [8]:
total    = len(gdf_final)
com_end  = int(gdf_final["tem_endereco"].sum())
sem_end  = total - com_end
com_cp7  = int(gdf_final["cp7"].notna().sum())
n1       = int((gdf_final["n_fragmentos"] == 1).sum())
nmulti   = int((gdf_final["n_fragmentos"] > 1).sum())

print("=" * 50)
print("RESUMO DO DISSOLVE POR NÚMERO DE POLÍCIA")
print("=" * 50)
print(f"Polígonos originais              : 88,649")
print(f"Edifícios reais resultantes      : {total:,}")
print(f"  Com endereço (nº polícia)      : {com_end:,} ({com_end/total*100:.1f}%)")
print(f"  Sem endereço (anexos/piscinas) : {sem_end:,} ({sem_end/total*100:.1f}%)")
print(f"  Edifício = 1 polígono          : {n1:,} ({n1/total*100:.1f}%)")
print(f"  Edifício = 2+ polígonos        : {nmulti:,} ({nmulti/total*100:.1f}%)")
print(f"Com CP7 atribuído                : {com_cp7:,} ({com_cp7/total*100:.1f}%)")
print(f"Fragmentos máx. por edifício     : {int(gdf_final['n_fragmentos'].max())}")
print()

# Distribuição de fragmentos
dist = Counter(gdf_final["n_fragmentos"].tolist())
print("Distribuição de fragmentos por edifício:")
for k in sorted(dist.keys())[:12]:
    print(f"  {k:2d} fragmento(s): {dist[k]:,}")

# Exportar
gdf_final.to_file("edificios_aveiro_npolicia.gpkg", driver="GPKG", layer="edificios_reais")
print("Ficheiro exportado: edificios_aveiro_npolicia.gpkg")


RESUMO DO DISSOLVE POR NÚMERO DE POLÍCIA
Polígonos originais              : 88,649
Edifícios reais resultantes      : 35,765
  Com endereço (nº polícia)      : 18,157 (50.8%)
  Sem endereço (anexos/piscinas) : 17,608 (49.2%)
  Edifício = 1 polígono          : 24,802 (69.3%)
  Edifício = 2+ polígonos        : 10,963 (30.7%)
Com CP7 atribuído                : 15,736 (44.0%)
Fragmentos máx. por edifício     : 64

Distribuição de fragmentos por edifício:
   1 fragmento(s): 24,802
   2 fragmento(s): 2,858
   3 fragmento(s): 1,769
   4 fragmento(s): 1,341
   5 fragmento(s): 1,014
   6 fragmento(s): 760
   7 fragmento(s): 616
   8 fragmento(s): 491
   9 fragmento(s): 406
  10 fragmento(s): 339
  11 fragmento(s): 235
  12 fragmento(s): 183
Ficheiro exportado: edificios_aveiro_npolicia.gpkg
